# Spam IX: Level 1, counter examples

We now play the **attacker** against every filter of the ladder. **Level 1** asks a simple question: *is there a cheap edit of a
spam message that the filter lets through?* A **counter example** is any such message; it need not still work as a scam. What
we want from it is a **pattern**: a blind spot of the model that we can name in one sentence.

> **Scope.** These are attacks on *our own* classifiers, in a lab, on the public SMS corpus. This is how robustness is measured;
> red teams use the same techniques and the results tell a defender what to fix. Level 2 (notebook 09) adds the constraints that
> make an example *useful* to a scammer, and notebook 10 the defences.

**Threat model** (state it before attacking: *no threat model, no security claim*):

| | this notebook |
|:--|:--|
| goal | integrity: a spam message classified as ham (*evasion*) |
| capability | append words, disguise characters (no other edit) |
| knowledge | white-box (weights) or black-box (only the score of a message: one **query** each) |
| stage | test time |

**Victims** (all trained on the training split, wrapped as `score(texts)`: higher = more spam, `score < 0` = inbox):
Naive Bayes on presence and on counts, TF-IDF + logistic regression, fastText (supervised), in-domain fastText vectors + LR,
MiniLM + LR and Nomic v2 MoE + LR (the last two run in llama.cpp servers, `make llama-up`; `AAS_SLM=skip` leaves them out and the notebook then needs no server).

**Metrics.** *ASR* (attack success rate): among the spam messages the filter catches, the share that evades after the attack.
*Queries*: how often the attacker asked the filter. *Edits*: words changed.

In [ ]:
import os
from collections import Counter

import numpy as np
from matplotlib import pyplot as plt

import attacklib as al
import spamlib as sl

plt.rcParams["figure.dpi"] = 100
texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)
spam_test = [t for t, lab in zip(X_test, y_test) if lab == 1]

WHICH = ["nb_presence", "nb_counts", "tfidf_lr", "fasttext", "vectors_lr"]
if os.environ.get("AAS_SLM") != "skip":
    WHICH += ["minilm_lr", "nomic_lr"]
try:
    V = al.build_victims(X_train, y_train, which=WHICH)
except RuntimeError as exc:  # a llama.cpp server is not running: leave the embedding victims out
    print("embedding victims skipped:", exc)
    V = al.build_victims(X_train, y_train, which=[k for k in WHICH if k not in ("minilm_lr", "nomic_lr")])
print(f"{'victim':26}{'spam caught':>12}")
for key, v in V.items():
    print(f"{v.name:26}{v.caught(spam_test).mean():12.3f}")

## A. Naive Bayes is an additive score

For a Multinomial Naive Bayes the log-odds of a message are **exactly**
$z=\log\frac{P(\text{spam})}{P(\text{ham})}+\sum_w c_w\,\text{llr}_w$ with $\text{llr}_w=\log\frac{P(w\mid\text{spam})}{P(w\mid\text{ham})}$ and
$c_w$ the count of the word (a 0/1 presence for the *presence* model). Every word contributes independently, so the attacker's
problem becomes: *add the words with the most negative* $\text{llr}$ *until the sum crosses zero.* Let us verify the identity.

In [ ]:
nb_models = {k: V[k] for k in ("nb_presence", "nb_counts")}
llr, prior, vocab = {}, {}, {}
for key, v in nb_models.items():
    vec, nb = v.encoder[0], v.encoder[1]
    llr[key] = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
    prior[key] = nb.class_log_prior_[1] - nb.class_log_prior_[0]
    vocab[key] = vec.get_feature_names_out()
    z = np.asarray(vec.transform(spam_test) @ llr[key]).ravel() + prior[key]
    print(f"{v.name:24} score == prior + counts . llr : {np.allclose(z, v.score(spam_test))}")
    order = np.argsort(llr[key])
    print("   most negative llr:", [(vocab[key][i], round(float(llr[key][i]), 2)) for i in order[:6]])

The five most "hammy" words include `lt` and `gt`: in this corpus they are the leftovers of `&lt;#&gt;`, an HTML-escaped placeholder
that only appears in *ham* messages (a data artefact). The model treats an accident of data collection as strong evidence, and an
attacker who reads the weights learns it for free.

**White-box closed form.** No search is needed: sort the words by llr and take the best ones. For the *counts* model the best word can
be **repeated**; for the *presence* model each word counts once. Fraction of the caught spam that flips with $k$ appended words:

In [ ]:
KS = [1, 2, 3, 5, 8, 12, 20]
print(f"{'k appended words':28}" + "".join(f"{k:>6}" for k in KS))
for key, v in nb_models.items():
    vec = v.encoder[0]
    z0 = np.asarray(vec.transform(spam_test) @ llr[key]).ravel() + prior[key]
    z0 = z0[z0 >= 0]
    best = np.sort(llr[key])
    if key == "nb_counts":  # repeat the best word
        row = [(z0 + k * best[0] < 0).mean() for k in KS]
    else:  # k distinct best words
        row = [(z0 + best[:k].sum() < 0).mean() for k in KS]
    print(f"{v.name:28}" + "".join(f"{r:6.2f}" for r in row))

Two findings, both of them *patterns*:

1. **Repetition beats the counts model** (and does nothing to the presence model): the count of a word multiplies its evidence.
   The presence model is capped by the number of distinct hammy words.
2. **Out-of-vocabulary padding is useless** against Naive Bayes: an unknown word has no weight. Check it: appending 20 unseen tokens
   changes the score by exactly 0.

In [ ]:
pad = " ".join(f"zzqx{i}" for i in range(20))
for key, v in nb_models.items():
    print(
        f"{v.name:24} score change after 20 unseen tokens: {abs(v.score([spam_test[0] + ' ' + pad]) - v.score([spam_test[0]]))[0]:.6f}"
    )

## B. Black-box: greedy search with queries

The white-box attacker read the weights. A **black-box** attacker can only *ask*: append every word of a candidate list, keep the one
that lowers the score most, repeat (a derivative-free search, as in the first class). The candidates come from **public knowledge**
(the 40 most frequent words of everyday chat), not from the victim. We attack the same first messages of every victim.

In [ ]:
ham_words = [w for t, lab in zip(X_train, y_train) if lab == 0 for w in sl.tokenize(t) if w.isalpha() and len(w) > 1]
uniq, counts = np.unique(ham_words, return_counts=True)
CANDIDATES = [str(w) for w in uniq[np.argsort(-counts)][:40]]
print("candidate pool:", CANDIDATES[:12], "...")

N_ATTACK = 20
subset = spam_test[:N_ATTACK]
BUDGETS = [0, 1, 2, 3, 5, 8, 12]
attacks = {}
print(f"{'victim':26}" + "".join(f"{b:>6}" for b in BUDGETS) + f"{'queries':>9}")
for key, v in V.items():
    res = [al.greedy(v, m, [al.op_append(CANDIDATES)], max_steps=12) for m in subset]
    attacks[key] = res
    steps = [r.steps if r.success else 99 for r in res]
    curve = [np.mean([s <= b for s in steps]) for b in BUDGETS]
    print(f"{v.name:26}" + "".join(f"{c:6.2f}" for c in curve) + f"{np.mean([r.queries for r in res]):9.0f}")

(Budget 0 = messages the filter already misses.) With **about a dozen harmless words** the search evades a large share of the
messages of *every* family, at a cost of a few hundred queries per message. Which words did it choose?

In [ ]:
for key, res in attacks.items():
    used = Counter(w for r in res if r.success for w in r.adversarial.split()[len(r.original.split()) :])
    print(f"{V[key].name:26}", [w for w, _ in used.most_common(6)])

The same handful of words (`gt`, `lt`, `ok`, `my`, `when`) works everywhere: they are frequent *and* only ever seen in ham.
**Pattern:** *the cheapest evidence to fake is a common chat word that never appears in spam.*

### The shortest counter example of each victim

In [ ]:
for key, res in attacks.items():
    wins = [r for r in res if r.success and r.steps > 0]
    if wins:
        r = min(wins, key=lambda r: (r.steps, len(r.original)))
        added = " ".join(r.adversarial.split()[len(r.original.split()) :])
        print(f"{V[key].name:26} +{r.steps} words: ...{r.original[-40:]!r} + [{added}]")

## C. Dilution: bury the spam in benign text

TF-IDF vectors are **L2-normalised**: appending words makes every spam word a smaller share of the vector. Mean-pooled embeddings
average over all tokens: benign text pulls the vector towards the ham region. Both suggest one pattern: *dilute*. Append $n$ random
ham messages to each spam message (no search, no queries beyond the test itself):

In [ ]:
rng = np.random.default_rng(1)
ham_pool = [t for t, lab in zip(X_train, y_train) if lab == 0 and 40 < len(t) < 100]
NS = [0, 1, 2, 4, 8]
print(f"{'ham messages appended':26}" + "".join(f"{n:>6}" for n in NS))
dilution = {}
for key, v in V.items():
    row = []
    for n in NS:
        pads = [" ".join(rng.choice(ham_pool, size=n, replace=False)) if n else "" for _ in spam_test]
        row.append(float((v.score([f"{m} {p}".strip() for m, p in zip(spam_test, pads)]) < 0).mean()))
    dilution[key] = row
    print(f"{v.name:26}" + "".join(f"{r:6.2f}" for r in row))

Word-count models (Naive Bayes, TF-IDF + LR, fastText and the mean of word vectors) collapse with a few benign messages: eight of them
evade almost everything. The **Transformer embedders resist much better** (attention and a trained pooling decide what matters, and
the 512-token window is far from full), but they are not immune. **Pattern:** *length is an attack surface for averaging models.*

## D. Characters: what the tokenizer does not see

Filters match **tokens**. Disguise the 25 most spammy words of the whole message and the tokenizer sees other strings:
`fr33` (leet), `f.r.e.e` (dots), `f r e e` (spaces), `fr` (vowels dropped), a Cyrillic `а` in `cаsh` (homoglyph), and a zero-width space
inside `fr​ee`. What does our tokenizer make of them?

In [ ]:
for word in ("free", "fr33", "f.r.e.e", "f r e e", "fr", "cаsh", "fr​ee"):
    print(f"{word!r:14} -> {sl.tokenize(word)}")

The homoglyph and the zero-width space **split or drop** characters; the word is gone for a bag of words. Sub-word models and
embedders see *some* of it. Recall on the disguised test spam (the words are the top weights of the logistic regression):

In [ ]:
lr_pipe = V["tfidf_lr"].encoder
feat = np.array(lr_pipe[0].get_feature_names_out())
TOP = [w for w in feat[np.argsort(-lr_pipe[1].coef_[0])] if w.isalpha()][:25]
print("disguised words:", TOP[:12], "...")
MODES = ["none", "leet", "vowel", "space", "dots", "homoglyph", "zwsp"]
disguise = {}
print(f"{'disguise':12}" + "".join(f"{v.name[:14]:>16}" for v in V.values()))
for mode in MODES:
    docs = spam_test if mode == "none" else [al.disguise_text(t, TOP, mode, protect=False) for t in spam_test]
    disguise[mode] = {k: float(v.caught(docs).mean()) for k, v in V.items()}
    print(f"{mode:12}" + "".join(f"{disguise[mode][k]:16.3f}" for k in V))

Reading the table (spam still **caught**; lower is better for the attacker):

* **Naive Bayes barely moves.** Its evidence is spread over hundreds of words, so disguising 25 leaves the rest; but it is the
  weakest against *appended* words (section B). No model is weak against everything: each has its **own cheapest attack**.
* **TF-IDF + LR and the vector models lose 10 to 25 points**, most with homoglyphs, spaces and dots (the token disappears).
* **fastText** is hurt most by dots (`f.r.e.e` leaves no useful n-gram: 0.915 to 0.698); the **SLM embedders** lose little (they were
  pre-trained on text with typos and odd characters): MiniLM most with dots (-7 points), Nomic most with the zero-width space (-6 points).
* The recall never collapses: spam is **redundant** (number, `£`, `!`, `call`, `txt`). A *level 2* attacker must keep those (notebook 09).

## E. A universal suffix

Instead of one search per message, look for **one short suffix that works for every spam message** (a *universal trigger*, Wallace et al.,
2019). Greedy search on 30 *training* spam messages, then apply it to the *test* spam.

In [ ]:
spam_dev = [t for t, lab in zip(X_train, y_train) if lab == 1][:30]
LEN = 8
universal = {}
print(f"{'victim':26}{'evasion (clean)':>17}{'evasion (suffix)':>18}   suffix")
for key, v in V.items():
    suffix = al.universal_trigger(v, spam_dev, CANDIDATES[:30], length=LEN)
    ev = float((v.score([f"{m} {' '.join(suffix)}" for m in spam_test]) < 0).mean())
    universal[key] = ev
    print(f"{v.name:26}{1 - v.caught(spam_test).mean():17.3f}{ev:18.3f}   {' '.join(suffix)}")

A fixed suffix of eight words, found *without seeing a single test message*, evades 16 to 43% of the test spam (5 to 9% without it). It is
weaker than the per-message search but costs nothing at attack time (no queries): it can be **copy-pasted into a whole campaign**.

## F. The catalogue: patterns against models

One table for the report: the share of the spam that evades ($=1-$recall) under each pattern, per victim.

In [ ]:
rows = {}
rows["clean (no attack)"] = {k: 1 - v.caught(spam_test).mean() for k, v in V.items()}
rows["12 words, greedy (black-box)"] = {k: np.mean([r.success and r.steps <= 12 for r in attacks[k]]) for k in V}
rows["4 ham messages appended"] = {k: dilution[k][NS.index(4)] for k in V}
for mode in ("leet", "dots", "homoglyph", "zwsp"):
    rows[f"disguise: {mode}"] = {k: 1 - disguise[mode][k] for k in V}
rows["universal 8-word suffix"] = universal
print(f"{'pattern':30}" + "".join(f"{v.name[:14]:>16}" for v in V.values()))
for name, row in rows.items():
    print(f"{name:30}" + "".join(f"{row[k]:16.2f}" for k in V))

## G. Transfer: attack one model, hit another

Do the messages crafted against one model also evade the others? (Row: crafted against; column: tested on; share that evades.)

In [ ]:
adv = {k: [r.adversarial for r in res] for k, res in attacks.items()}
tm = al.transfer_matrix(adv, V)
print(f"{'crafted against':26}" + "".join(f"{v.name[:14]:>16}" for v in V.values()))
for src, row in tm.items():
    print(f"{V[src].name:26}" + "".join(f"{row[dst]:16.2f}" for dst in V))

The diagonal is high (the attack was optimised for that model) and the rest is low: a **greedy per-message attack transfers poorly**,
mostly between similar models (the word-vector model inherits Naive Bayes' and TF-IDF's weaknesses). The embedders are the least
affected by attacks crafted elsewhere. That is not robustness: it is that **the attacker optimised for another model**. An attacker who
can query the target does not need transfer (section B); one who cannot must first *steal* the model (notebook 10 exercises).

## Take-aways

* **Accuracy is not security.** All victims score F1 above 0.93 on clean data, and every one of them has a cheap counter example.
* Each family has its **own cheapest pattern**: repetition and appended hammy words for Naive Bayes, dilution and disguise for
  TF-IDF and vector models, zero-width characters and very long benign text for the embedders.
* **Reading the weights is the strongest attack** (closed form for Naive Bayes); black-box search needs only queries.
* Some blind spots are **data artefacts** (`gt`, `lt`): the model learnt an accident, the attacker finds it for free.

## Exercises

1. In B replace the greedy search by `al.random_search`. How many more words are needed, and what does that say about the value of
   the score feedback?
2. Give the attacker the *victim's own* best words (the closed form of A) as the candidate list of the greedy search on Naive Bayes. Does it
   match the closed form? What is the difference between white-box and black-box knowledge in queries and in words?
3. Explain, with the tokenizer output of D, why a zero-width space hurts a bag-of-words model but much less a Transformer embedder.
4. Add one row to the catalogue: append `gt lt` **three times**. Which victims break, and why?